In [7]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✅ All libraries imported successfully")

✅ All libraries imported successfully


In [12]:
# Load dataset
df = pd.read_csv('Social_Media_Advertising.csv')  

# Check if loaded correctly
print(f"Dataset Shape: {df.shape}")
print(f"\nTotal Rows: {df.shape[0]:,}")
print(f"Total Columns: {df.shape[1]}")
print("\nFirst 5 rows:")
df.head()

Dataset Shape: (300000, 16)

Total Rows: 300,000
Total Columns: 16

First 5 rows:


,Campaign_ID,Target_Audience,Campaign_Goal,Duration,Channel_Used,Conversion_Rate,Acquisition_Cost,ROI,Location,Language,Clicks,Impressions,Engagement_Score,Customer_Segment,Date,Company
0,529013,Men 35-44,Product Launch,15 Days,Instagram,0.15,$500.00,5.790000,Las Vegas,Spanish,500,3000,7,Health,2022-02-25,Aura Align
1,275352,Women 45-60,Market Expansion,15 Days,Facebook,0.01,$500.00,7.210000,Los Angeles,French,500,3000,5,Home,2022-05-12,Hearth Harmony
2,692322,Men 45-60,Product Launch,15 Days,Instagram,0.08,$500.00,0.430000,Austin,Spanish,500,3000,9,Technology,2022-06-19,Cyber Circuit
3,675757,Men 25-34,Increase Sales,15 Days,Pinterest,0.03,$500.00,0.909824,Miami,Spanish,293,1937,1,Health,2022-09-08,Well Wish
4,535900,Men 45-60,Market Expansion,15 Days,Pinterest,0.13,$500.00,1.422828,Austin,French,293,1937,1,Home,2022-08-24,Hearth Harmony


In [14]:
# Basic info about dataset
print("Column Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nBasic Statistics:")
df.describe()

Column Names:
['Campaign_ID', 'Target_Audience', 'Campaign_Goal', 'Duration', 'Channel_Used', 'Conversion_Rate', 'Acquisition_Cost', 'ROI', 'Location', 'Language', 'Clicks', 'Impressions', 'Engagement_Score', 'Customer_Segment', 'Date', 'Company']

Data Types:
Campaign_ID           int64
Target_Audience      object
Campaign_Goal        object
Duration             object
Channel_Used         object
Conversion_Rate     float64
Acquisition_Cost     object
ROI                 float64
Location             object
Language             object
Clicks                int64
Impressions           int64
Engagement_Score      int64
Customer_Segment     object
Date                 object
Company              object
dtype: object

Basic Statistics:


,Campaign_ID,Conversion_Rate,ROI,Clicks,Impressions,Engagement_Score
count,300000.000000,300000.000000,300000.000000,300000.000000,300000.000000,300000.000000
mean,550444.804487,0.080009,3.177691,18153.670370,56034.236387,4.369217
std,260252.586037,0.040563,2.461200,11027.023294,32583.136334,3.156492
min,100001.000000,0.010000,0.000000,293.000000,1937.000000,1.000000
25%,325003.500000,0.050000,0.930000,8821.000000,28362.000000,1.000000
50%,551164.500000,0.080000,2.670000,17230.000000,54098.000000,4.000000
75%,776284.500000,0.110000,5.330000,26808.000000,80925.250000,7.000000
max,999998.000000,0.150000,8.000000,40000.000000,120000.000000,10.000000


In [16]:
# Check for missing values
print("Missing Values:")
missing = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_percent
})
print(missing_df[missing_df['Missing Count'] > 0])

print(f"\nDuplicate Rows: {df.duplicated().sum()}")

Missing Values:
Empty DataFrame
Columns: [Missing Count, Percentage]
Index: []

Duplicate Rows: 0


In [18]:
# Check unique values in each categorical column
categorical_cols = ['Target_Audience', 'Campaign_Goal', 'Duration', 
                    'Channel_Used', 'Location', 'Language', 
                    'Customer_Segment', 'Company']

for col in categorical_cols:
    print(f"\n{col} ({df[col].nunique()} unique values):")
    print(df[col].value_counts().head(10))
    print("-" * 40)


Target_Audience (9 unique values):
Target_Audience
Women 18-24    33593
Men 45-60      33491
Women 25-34    33482
All Ages       33447
Men 25-34      33346
Men 35-44      33204
Men 18-24      33181
Women 35-44    33142
Women 45-60    33114
Name: count, dtype: int64
----------------------------------------

Campaign_Goal (4 unique values):
Campaign_Goal
Brand Awareness     75248
Product Launch      75030
Increase Sales      74963
Market Expansion    74759
Name: count, dtype: int64
----------------------------------------

Duration (4 unique values):
Duration
15 Days    75000
30 Days    75000
45 Days    75000
60 Days    75000
Name: count, dtype: int64
----------------------------------------

Channel_Used (4 unique values):
Channel_Used
Facebook     75164
Instagram    75101
Pinterest    75082
Twitter      74653
Name: count, dtype: int64
----------------------------------------

Location (5 unique values):
Location
Los Angeles    60322
Miami          60241
New York       59859
Austin    

In [20]:
# Fix Acquisition_Cost - remove $ sign and convert to float
df['Acquisition_Cost'] = df['Acquisition_Cost'].str.replace('$', '').str.replace(',', '').astype(float)

# Fix Date column
df['Date'] = pd.to_datetime(df['Date'])

# Fix Duration - extract number of days
df['Duration_Days'] = df['Duration'].str.extract('(\d+)').astype(int)

print("✅ Data types fixed")
print(df[['Acquisition_Cost', 'Date', 'Duration_Days']].head())

✅ Data types fixed
   Acquisition_Cost       Date  Duration_Days
0             500.0 2022-02-25             15
1             500.0 2022-05-12             15
2             500.0 2022-06-19             15
3             500.0 2022-09-08             15
4             500.0 2022-08-24             15


In [22]:
# 1. Click Through Rate (CTR)
df['CTR'] = (df['Clicks'] / df['Impressions']) * 100

# 2. Cost Per Click (CPC)
df['CPC'] = df['Acquisition_Cost'] / df['Clicks']

# 3. Cost Per Impression (CPM)
df['CPM'] = (df['Acquisition_Cost'] / df['Impressions']) * 1000

# 4. Revenue Estimate
df['Revenue'] = df['Acquisition_Cost'] * (1 + df['ROI'])

# 5. ROI Category
df['ROI_Category'] = pd.cut(df['ROI'], 
                             bins=[0, 1, 3, 5, 8],
                             labels=['Low', 'Medium', 'High', 'Very High'])

# 6. Campaign Success (target variable for ML)
# Success = Conversion Rate above median
median_conversion = df['Conversion_Rate'].median()
df['Campaign_Success'] = (df['Conversion_Rate'] > median_conversion).astype(int)

# 7. Extract date features
df['Month'] = df['Date'].dt.month
df['Day_of_Week'] = df['Date'].dt.day_name()
df['Quarter'] = df['Date'].dt.quarter

print("✅ New features created")
print(f"\nMedian Conversion Rate: {median_conversion:.3f}")
print(f"\nCampaign Success Distribution:")
print(df['Campaign_Success'].value_counts())
print(f"\nROI Category Distribution:")
print(df['ROI_Category'].value_counts())

✅ New features created

Median Conversion Rate: 0.080

Campaign Success Distribution:
Campaign_Success
0    160792
1    139208
Name: count, dtype: int64

ROI Category Distribution:
ROI_Category
Very High    84267
Low          80546
Medium       78744
High         56258
Name: count, dtype: int64


In [26]:

import os

# Save in same directory as notebook
df.to_csv('cleaned_campaign_data.csv', index=False)
print("✅ Cleaned dataset saved")
print(f"\nFinal Dataset Shape: {df.shape}")
print(f"\nNew Columns Added:")
new_cols = ['CTR', 'CPC', 'CPM', 'Revenue', 'ROI_Category', 
            'Campaign_Success', 'Month', 'Day_of_Week', 
            'Quarter', 'Duration_Days']
for col in new_cols:
    print(f"  ✅ {col}")

✅ Cleaned dataset saved

Final Dataset Shape: (300000, 26)

New Columns Added:
  ✅ CTR
  ✅ CPC
  ✅ CPM
  ✅ Revenue
  ✅ ROI_Category
  ✅ Campaign_Success
  ✅ Month
  ✅ Day_of_Week
  ✅ Quarter
  ✅ Duration_Days


In [28]:
# Final check on cleaned data
print("Dataset Summary After Cleaning:")
print(f"Shape: {df.shape}")
print(f"\nSample of new features:")
df[['Campaign_ID', 'CTR', 'CPC', 'CPM', 'Revenue', 
    'ROI_Category', 'Campaign_Success']].head(10)

Dataset Summary After Cleaning:
Shape: (300000, 26)

Sample of new features:


,Campaign_ID,CTR,CPC,CPM,Revenue,ROI_Category,Campaign_Success
0,529013,16.666667,1.000000,166.666667,3395.000000,Very High,1
1,275352,16.666667,1.000000,166.666667,4105.000000,Very High,0
2,692322,16.666667,1.000000,166.666667,715.000000,Low,0
3,675757,15.126484,1.706485,258.131131,954.911786,Low,0
4,535900,15.126484,1.706485,258.131131,1211.414109,Medium,1
5,323031,16.661113,1.000000,166.611130,3950.000000,Very High,0
6,727501,15.118679,1.706485,257.997936,839.619801,Low,1
7,289553,15.118679,1.706485,257.997936,1004.961020,Medium,1
8,942511,16.683317,0.998004,166.500167,1095.000000,Medium,1
9,255854,16.683317,0.998004,166.500167,1325.000000,Medium,0
